# SML Fine-tuning on Kaggle (2× T4)

LoRA fine-tuning of `Qwen/Qwen3.5-4B` on internal document QA pairs using [ms-swift](https://swift.readthedocs.io/en/latest/GetStarted/SWIFT-installation.html).  
Training runs on 2× NVIDIA T4 GPUs (Kaggle free tier) with `CUDA_VISIBLE_DEVICES=0,1`.

This guide consists of 3 main steps:

> **[Step 1: Convert Data to Training Format](#step-1-convert-data-to-training-format)**

> **[Step 2: LoRA Finetuning with Swift](#step-2-lora-finetuning-with-swift)**

> **[Step 3: Merge LoRA Adapter & Upload to Hugging Face](#step-3-merge-lora-adapter--upload-to-hugging-face)**

## Install Dependencies

In [ ]:
!pip install -q uv

!uv pip install -U \
  ms-swift \
  wandb \
  "transformers>=4.56.2" \
  "peft==0.11.1" \
  "qwen_vl_utils>=0.0.14" \
  hf

In [ ]:
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0
!pip uninstall -y cuml-cu12 cudf-cu12 libcuvs-cu12 libraft-cu12

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import wandb
wandb.login()
run = wandb.init(project="sml_training")

In [18]:
import sys
import importlib
from packaging import version

def check(pkg_name, min_ver=None, max_ver=None):
    try:
        module = importlib.import_module(pkg_name)
        ver = getattr(module, "__version__", "unknown")
        
        ok = True
        if min_ver and version.parse(ver) < version.parse(min_ver):
            ok = False
        if max_ver and version.parse(ver) >= version.parse(max_ver):
            ok = False

        status = "✅ OK" if ok else "❌ NOT OK"
        print(f"{pkg_name:15} {ver:15} {status}")
    except ImportError:
        print(f"{pkg_name:15} NOT INSTALLED ❌")

# Python
py_ver = sys.version.split()[0]
print(f"{'python':15} {py_ver:15} {'✅ OK' if version.parse(py_ver)>=version.parse('3.10') else '❌ NOT OK'}")

# CUDA (via torch)
try:
    import torch
    cuda_ver = torch.version.cuda
    print(f"{'cuda':15} {cuda_ver:15} (from torch)")
except:
    print(f"{'cuda':15} NOT AVAILABLE")

print("\n--- Package Checks ---")

check("torch", "2.0")
check("transformers", "4.33")
check("modelscope", "1.23")
check("peft", "0.11", "0.20")
check("flash_attn")  # version varies, just check install
check("trl", "0.15", "0.30")
check("deepspeed", "0.14")

python          3.12.12         ✅ OK
cuda            12.8            (from torch)

--- Package Checks ---
torch           2.10.0+cu128    ✅ OK
transformers    5.0.0           ✅ OK
modelscope      1.36.3          ✅ OK
peft            0.18.1          ✅ OK
flash_attn      NOT INSTALLED ❌
trl             0.29.1          ✅ OK
deepspeed       NOT INSTALLED ❌


## Step 1: Convert Data to Training Format

In [2]:
"""
Convert QA pairs generated from internal documents to ms-swift training format.

Input (JSONL from qa_generator.py):
{
    "source_file": "...",
    "breadcrumb": "...",
    "heading": "...",
    "context": "...",
    "question": "...",
    "answer": "..."
}

Output (ms-swift JSONL):
{
    "messages": [
        {"role": "system", "content": "..."},
        {"role": "user", "content": "..."},
        {"role": "assistant", "content": "..."}
    ]
}
"""
import json
import os
import argparse
from pathlib import Path


SYSTEM_PROMPT = """You are a helpful assistant. Answer questions accurately, clearly, and concisely."""


def convert_to_swift_format(input_file: str, output_file: str) -> int:
    print(f"Loading data from {input_file}...")

    data = []
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))

    print(f"Loaded {len(data)} QA pairs")

    converted, skipped = [], 0

    for item in data:
        if not item.get("question") or not item.get("answer"):
            skipped += 1
            continue

        converted.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": item["question"].strip()},
                {"role": "assistant", "content": item["answer"].strip()}
            ]
        })

    print(f"Skipped {skipped} invalid samples")
    print(f"Valid samples: {len(converted)}")

    print(f"Saving to {output_file}...")
    with open(output_file, 'w', encoding='utf-8') as f:
        for item in converted:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    print(f"Done: {len(converted)} samples saved to {output_file}")
    return len(converted)


def collect_qa_files(base_directory: str) -> list:
    """Collect all _qa.jsonl files from batch folder structure."""
    return sorted(Path(base_directory).rglob('*_qa.jsonl'))



In [7]:
qa_files = collect_qa_files("/kaggle/input/datasets/trnnamnhtanh/sml-training-dataset/output_batches")
if not qa_files:
    print(f"No *_qa.jsonl files found in {"/kaggle/input/datasets/trnnamnhtanh/sml-training-dataset/output_batches"}")
    exit(1)

print(f"Found {len(qa_files)} QA files, merging...")
merged_tmp = Path("/kaggle/working/training_data.jsonl").with_suffix('.tmp.jsonl')

with open(merged_tmp, 'w', encoding='utf-8') as out_f:
    for qa_file in qa_files:
        with open(qa_file, 'r', encoding='utf-8') as in_f:
            for line in in_f:
                if line.strip():
                    out_f.write(line)

count = convert_to_swift_format(str(merged_tmp), "/kaggle/working/training_data.jsonl")
merged_tmp.unlink()

print(f"\nDataset ready: {"/kaggle/working/training_data.jsonl"} ({count} samples)")


Found 5 QA files, merging...
Loading data from /kaggle/working/training_data.tmp.jsonl...
Loaded 862 QA pairs
Skipped 711 invalid samples
Valid samples: 151
Saving to /kaggle/working/training_data.jsonl...
Done: 151 samples saved to /kaggle/working/training_data.jsonl

Dataset ready: /kaggle/working/training_data.jsonl (151 samples)


In [9]:
import random

print("\nRandom 5 samples:\n")

with open("/kaggle/working/training_data.jsonl", 'r', encoding='utf-8') as f:
    samples = [json.loads(line) for line in f if line.strip()]

random_samples = random.sample(samples, min(5, len(samples)))

for i, sample in enumerate(random_samples, 1):
    print(f"{'=' * 80}")
    print(f"Random Sample #{i}")
    print(f"{'=' * 80}")

    for msg in sample["messages"]:
        print(f"\n[{msg['role'].upper()}]")
        print(msg["content"])

    print("\n")


Random 5 samples:

Random Sample #1

[SYSTEM]
You are a helpful assistant. Answer questions accurately, clearly, and concisely.

[USER]
According to the guideline, when is it necessary to involve an independent third party in the decision‑making process?

[ASSISTANT]
It is necessary to involve an independent third party when you cannot limit your participation in the conflict of interest or when such limitation does not resolve the possible conflict.


Random Sample #2

[SYSTEM]
You are a helpful assistant. Answer questions accurately, clearly, and concisely.

[USER]
What types of personal information does The Company collect from its staff?

[ASSISTANT]
The Company collects personal information from its staff that includes labor status, salary, bonus, health status, and other incentives.


Random Sample #3

[SYSTEM]
You are a helpful assistant. Answer questions accurately, clearly, and concisely.

[USER]
What measures does FPT Software implement to protect personal information from m

## Step 2: LoRA Finetuning with Swift
Runs supervised fine-tuning with LoRA on `Qwen/Qwen3.5-4B`. Key configuration:

| Parameter | Value |
|---|---|
| LoRA rank / alpha | 32 / 64 |
| Target modules | `q_proj k_proj v_proj o_proj gate_proj up_proj down_proj` |
| Batch size | 4 × 4 gradient accumulation = effective batch 16 |
| Max sequence length | 512 |
| Learning rate | 2e-4 with 5% warmup |
| Epochs | 10 |
| Attention | `sdpa` (flash_attention unavailable on T4) |
| Eval / save every | 30 steps |

In [24]:
 !CUDA_VISIBLE_DEVICES=0,1 swift sft \
    --model Qwen/Qwen3.5-4B \
    --use_hf true \
    --dataset /kaggle/input/datasets/trnnamnhtanh/sml-training-dataset/output_batches/training_data.jsonl \
    --tuner_type lora \
    --lora_rank 32 \
    --lora_alpha 64 \
    --lora_dropout 0.05 \
    --target_modules q_proj k_proj v_proj,o_proj gate_proj up_proj down_proj \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --max_length 512 \
    --learning_rate 2e-4 \
    --warmup_ratio 0.05 \
    --num_train_epochs 10 \
    --attn_impl sdpa \
    --output_dir /kaggle/working/sml_training/qwen35 \
    --save_strategy steps \
    --early_stop_interval 3 \
    --eval_steps 30 \
    --save_steps 30 \
    --dataloader_num_workers 2 \
    --dataset_num_proc 8 \
    --load_from_cache_file true \
    --model_author swift \
    --model_name swift-robot \
    --loss_scale default \
    --report_to wandb

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/sft.py --model Qwen/Qwen3.5-4B --use_hf true --dataset /kaggle/input/datasets/trnnamnhtanh/sml-training-dataset/output_batches/training_data.jsonl --tuner_type lora --lora_rank 32 --lora_alpha 64 --lora_dropout 0.05 --target_modules q_proj k_proj v_proj,o_proj gate_proj up_proj down_proj --per_device_train_batch_size 4 --gradient_accumulation_steps 4 --max_length 512 --learning_rate 2e-4 --warmup_ratio 0.05 --num_train_epochs 10 --attn_impl sdpa --output_dir /kaggle/working/sml_training/qwen35 --save_strategy steps --early_stop_interval 3 --eval_steps 30 --save_steps 30 --dataloader_num_workers 2 --dataset_num_proc 8 --load_from_cache_file true --model_author swift --model_name swift-robot --loss_scale default --report_to wandb`
[INFO:swift] Successfully registered `/usr/local/lib/python3.12/dist-packages/swift/dataset/data/dataset_info.json`.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_si

## Step 3: Merge Lora Adapter & Upload to Huggingface Hub

In [25]:
!swift export --adapters /kaggle/working/sml_training/qwen35/v0-20260508-064446/checkpoint-100 --merge_lora true --use_hf true

run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/export.py --adapters /kaggle/working/sml_training/qwen35/v0-20260508-064446/checkpoint-100 --merge_lora true --use_hf true`
[INFO:swift] Successfully registered `/usr/local/lib/python3.12/dist-packages/swift/dataset/data/dataset_info.json`.
[INFO:swift] Successfully loaded /kaggle/working/sml_training/qwen35/v0-20260508-064446/checkpoint-100/args.json.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[INFO:swift] Downloading the model from HuggingFace Hub, model_id: Qwen/Qwen3.5-4B
Fetching 12 files: 100%|█████████████████████| 12/12 [00:00<00:00, 11006.26it/s]
Download complete: : 0.00B [00:00, ?B/s]              [INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--Qwen--Qwen3.5-4B/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[WARNING:swift] Please install the package: `pip install "d

In [26]:
!hf repo create tnnanh1005/Qwen3.5-4B-SML-ver2 --repo-type model
!hf upload tnnanh1005/Qwen3.5-4B-SML-ver2 /kaggle/working/sml_training/qwen35/v0-20260508-064446/checkpoint-100-merged --repo-type model

✓ Repo created
  repo_id: tnnanh1005/Qwen3.5-4B-SML-ver2
  url: https://huggingface.co/tnnanh1005/Qwen3.5-4B-SML-ver2
Start hashing 11 files.
Finished hashing 11 files.
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...100-merged/tokenizer.json: 100%|██████████████| 20.0MB / 20.0MB            


  ...0001-of-00002.safetensors:   3%|▍             |  160MB / 4.97GB            

  ...100-merged/tokenizer.json: 100%|██████████████| 20.0MB / 20.0MB            


Processing Files (1 / 3)      :   2%|▎             |  180MB / 9.10GB,   ???B/s  

  ...100-merged/tokenizer.json: 100%|██████████████| 20.0MB / 20.0MB            


Processing Files (1 / 3)      :   4%|▌             |  356MB / 9.10GB,   ???B/s  



  ...0002-of-00002.safetensors:   1%|▏             | 49.5MB / 4.11GB            

  ...100-merged/tokenizer.json: 100%|██████████████| 20.0MB / 20.0MB            


  ...0